In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import f1_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# ---------------------------- 工具函数 ----------------------------
def compute_pairwise_se_distance(X, Y):

    XX = np.sum(X**2, axis=1, keepdims=True)
    YY = np.sum(Y**2, axis=1, keepdims=True).T
    XY = np.dot(X, Y.T)
    return XX + YY - 2*XY

def compute_centroids(opinions, labels, K):

    centroids = np.zeros((K, opinions.shape[1]))
    for k in range(K):
        mask = (labels == k)
        if np.any(mask):
            centroids[k] = opinions[mask].mean(axis=0)
        else:
            centroids[k] = np.zeros(opinions.shape[1])  # 空簇质心为0
    return centroids

def avg_f1_score(true_labels, pred_labels, true_K=None, pred_K=None):

    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)
    if true_K is None:
        true_K = len(np.unique(true_labels))
    if pred_K is None:
        pred_K = len(np.unique(pred_labels))


    true_onehot = np.eye(true_K)[true_labels]
    pred_onehot = np.eye(pred_K)[pred_labels]

    sum_pred = 0.0
    for p in range(pred_K):
        f1_scores = []
        for t in range(true_K):
            f1 = f1_score(true_onehot[:, t], pred_onehot[:, p], zero_division=0)
            f1_scores.append(f1)
        sum_pred += max(f1_scores)

    sum_true = 0.0
    for t in range(true_K):
        f1_scores = []
        for p in range(pred_K):
            f1 = f1_score(true_onehot[:, t], pred_onehot[:, p], zero_division=0)
            f1_scores.append(f1)
        sum_true += max(f1_scores)

    avgf1 = (sum_pred / (2 * pred_K)) + (sum_true / (2 * true_K))
    return avgf1

# ---------------------------- GK-means 算法类 ----------------------------
class GKMeans:
    def __init__(self, K=2, d=64, beta=None, rho=0.5, epsilon=None,
                 max_iter_phase1=100, max_iter_phase3=40, random_state=42):

        self.K = K
        self.d = d
        self.beta = beta if beta is not None else d
        self.rho = rho
        self.epsilon = epsilon if epsilon is not None else d * 0.001
        self.max_iter_phase1 = max_iter_phase1
        self.max_iter_phase3 = max_iter_phase3
        self.random_state = random_state
        np.random.seed(random_state)

    def fit(self, G):

        n = G.number_of_nodes()
        m = G.number_of_edges()
        nodes = list(G.nodes())

        adj = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
        degrees = adj.sum(axis=1).astype(int)


        neighbors = [set(G.neighbors(i)) for i in nodes]


        LN_matrix = np.zeros((n, n))
        for i in range(n):
            for j in neighbors[i]:
                if i >= j:
                    continue
                common = len(neighbors[i] & neighbors[j])
                ln_val = (2 * common) / (degrees[i] * degrees[j]) if degrees[i]*degrees[j] > 0 else 0
                LN_matrix[i, j] = ln_val
                LN_matrix[j, i] = ln_val

        # ---------- Phase 1: Opinion Dynamics ----------

        opinions = np.random.rand(n, self.d)

        for t in range(self.max_iter_phase1):
            opinions_new = np.zeros_like(opinions)
            max_change = 0.0
            for i in range(n):

                dist_to_i = np.sum((opinions[list(neighbors[i])] - opinions[i])**2, axis=1)
                threshold = self.beta * (self.rho ** t)
                valid_mask = dist_to_i <= threshold
                valid_neighbors = [list(neighbors[i])[idx] for idx, v in enumerate(valid_mask) if v]

                if len(valid_neighbors) == 0:
                    opinions_new[i] = opinions[i]
                    alpha_tilde = 1.0
                else:

                    ln_vals = np.array([LN_matrix[i, j] for j in valid_neighbors])
                    w = ln_vals / ln_vals.sum()

                    alpha_tilde = np.random.rand()

                    weighted_opinions = np.sum(w[:, np.newaxis] * opinions[valid_neighbors], axis=0)
                    opinions_new[i] = alpha_tilde * opinions[i] + (1 - alpha_tilde) * weighted_opinions


                change = np.sum((opinions_new[i] - opinions[i])**2)
                if change > max_change:
                    max_change = change

            opinions = opinions_new
            if max_change < self.epsilon:
                break

        # ---------- Phase 2: Opinion Leader Identification ----------

        R = np.zeros(n)
        for i in range(n):
            for j in neighbors[i]:

                union = len(neighbors[i] | neighbors[j])
                jc = len(neighbors[i] & neighbors[j]) / union if union > 0 else 0

                dist_se = np.sum((opinions[i] - opinions[j])**2)
                R[i] += jc * np.exp(-dist_se)


        sorted_nodes = sorted(range(n), key=lambda x: R[x], reverse=True)


        labels = -np.ones(n, dtype=int)
        visited = np.zeros(n, dtype=bool)
        leader_count = 0
        for i in sorted_nodes:
            if leader_count >= self.K:
                break
            if not visited[i]:
                labels[i] = leader_count
                visited[i] = True
                for j in neighbors[i]:
                    visited[j] = True
                leader_count += 1


        unassigned = np.where(labels == -1)[0]
        if len(unassigned) > 0:

            leader_centroids = []
            for k in range(self.K):
                mask = (labels == k)
                if np.any(mask):
                    leader_centroids.append(opinions[mask].mean(axis=0))
                else:
                    leader_centroids.append(np.zeros(self.d))
            leader_centroids = np.array(leader_centroids)
            for i in unassigned:
                dists = np.sum((leader_centroids - opinions[i])**2, axis=1)
                labels[i] = np.argmin(dists)

        # ---------- Phase 3: Finding Locally Pareto Optimality ----------
        labels = labels.astype(int)
        for tau in range(self.max_iter_phase3):

            theta = np.zeros(self.K)
            for k in range(self.K):
                mask = (labels == k)
                theta[k] = degrees[mask].sum() / (2 * m) if m > 0 else 0

            eta = np.zeros((n, self.K))
            for i in range(n):
                if degrees[i] == 0:
                    continue
                neigh_labels = labels[list(neighbors[i])]
                for k in range(self.K):
                    eta[i, k] = np.sum(neigh_labels == k) / degrees[i]


            new_labels = labels.copy()
            for i in range(n):
                current_label = labels[i]
                feasible_set = []

                for p in range(self.K):
                    if p == current_label:
                        feasible_set.append(p)
                        continue
                    theta_p_new = theta[p] + degrees[i] / (2 * m)
                    theta_q_new = theta[current_label] - degrees[i] / (2 * m)

                    def L_i(label, theta_val, eta_val):
                        return - (degrees[i] / (2 * m)) * (eta_val - theta_val)

                    L_i_p = L_i(p, theta_p_new, eta[i, p])
                    L_i_q = L_i(current_label, theta_q_new, eta[i, current_label])

                    if 2 * (L_i_p - L_i_q) <= 0:
                        feasible_set.append(p)

                if not feasible_set:
                    feasible_set = [current_label]

                best_p = current_label
                best_dist = np.inf
                for p in feasible_set:
                    temp_labels = labels.copy()
                    temp_labels[i] = p
                    mask = (temp_labels == p)
                    if np.any(mask):
                        centroid = opinions[mask].mean(axis=0)
                    else:
                        centroid = np.zeros(self.d)
                    dist = np.sum((opinions[i] - centroid)**2)
                    if dist < best_dist:
                        best_dist = dist
                        best_p = p
                new_labels[i] = best_p

            if np.array_equal(new_labels, labels):
                break
            labels = new_labels

        return labels

# ---------------------------- 实验运行 ----------------------------
if __name__ == "__main__":

    W = np.zeros((105, 105))
    edges = pd.read_csv('PolBK.mtx', header=None, sep=' ')
    edges = edges - 1
    row = edges[0]
    col = edges[1]
    for i in range(len(row)):
        W[row[i]][col[i]] = 1
        W[col[i]][row[i]] = 1

    G = nx.from_numpy_array(W)
    n = G.number_of_nodes()
    print(f"节点数: {n}, 边数: {G.number_of_edges()}")

    # 构建真实标签
    COMMUNITY_LABELS_1 = np.array([1,5,7,8,19,29,47,49,52,70,77,104,105]) - 1
    COMMUNITY_LABELS_2 = np.array([31, 32,
                                   60, 61, 62, 63, 64, 65, 66, 67, 68, 69,
                                   71, 72, 73, 74, 75, 76, 79,
                                   80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
                                   90, 91, 92, 93, 94, 95, 96, 97, 98, 99,
                                   100, 101, 102, 103
                                  ]) - 1

    true_labels = np.zeros(n, dtype=int)
    true_labels[list(COMMUNITY_LABELS_1)] = 1
    true_labels[list(COMMUNITY_LABELS_2)] = 2


    K = 3
    d = 64
    beta = d          # β = d
    rho = 0.5
    epsilon = d * 0.001
    max_iter_phase1 = 100
    max_iter_phase3 = 30
    n_runs = 20

    # 存储结果
    all_pred_labels = []
    all_avgf1 = []
    all_nmi = []

    for run in range(n_runs):
        print(f"运行第 {run+1}/{n_runs} 次...")
        gk = GKMeans(K=K, d=d, beta=beta, rho=rho, epsilon=epsilon,
                     max_iter_phase1=max_iter_phase1, max_iter_phase3=max_iter_phase3,
                     random_state=run)
        pred = gk.fit(G)
        all_pred_labels.append(pred)

        # 计算指标
        avgf1 = avg_f1_score(true_labels, pred, true_K=K, pred_K=K)
        nmi = normalized_mutual_info_score(true_labels, pred)
        all_avgf1.append(avgf1)
        all_nmi.append(nmi)
        print(f"  AvgF1 = {avgf1:.4f}, NMI = {nmi:.4f}")

    # 输出 Excel 表格1: 20次聚类结果
    df_results = pd.DataFrame(np.array(all_pred_labels).T, columns=[f"Run_{i+1}" for i in range(n_runs)])
    df_results.to_excel("PolBK_clustering_results_20runs.xlsx", index=False)
    print("表格1已保存: PolBK_clustering_results_20runs.xlsx")

    # 输出 Excel 表格2: 每次的 AvgF1, NMI 及平均值
    metrics_df = pd.DataFrame({
        "Run": np.arange(1, n_runs+1),
        "AvgF1": all_avgf1,
        "NMI": all_nmi
    })
    avg_avgf1 = np.mean(all_avgf1)
    avg_nmi = np.mean(all_nmi)
    # 添加平均行
    mean_row = pd.DataFrame({"Run": ["Average"], "AvgF1": [avg_avgf1], "NMI": [avg_nmi]})
    metrics_df = pd.concat([metrics_df, mean_row], ignore_index=True)
    metrics_df.to_excel("PolBK_metrics_20runs.xlsx", index=False)
    print("表格2已保存: PolBK_metrics_20runs.xlsx")
    print(f"最终平均 AvgF1: {avg_avgf1:.4f}, 平均 NMI: {avg_nmi:.4f}")

节点数: 105, 边数: 441
运行第 1/20 次...
  AvgF1 = 0.6462, NMI = 0.4848
运行第 2/20 次...
  AvgF1 = 0.6419, NMI = 0.5268
运行第 3/20 次...
  AvgF1 = 0.6563, NMI = 0.5059
运行第 4/20 次...
  AvgF1 = 0.6293, NMI = 0.4484
运行第 5/20 次...
  AvgF1 = 0.7024, NMI = 0.5010
运行第 6/20 次...
  AvgF1 = 0.6711, NMI = 0.4389
运行第 7/20 次...
  AvgF1 = 0.6548, NMI = 0.5497
运行第 8/20 次...
  AvgF1 = 0.6089, NMI = 0.3190
运行第 9/20 次...
  AvgF1 = 0.6462, NMI = 0.4848
运行第 10/20 次...
  AvgF1 = 0.6527, NMI = 0.5542
运行第 11/20 次...
  AvgF1 = 0.6397, NMI = 0.4939
运行第 12/20 次...
  AvgF1 = 0.6561, NMI = 0.5128
运行第 13/20 次...
  AvgF1 = 0.6371, NMI = 0.5091
运行第 14/20 次...
  AvgF1 = 0.6462, NMI = 0.4848
运行第 15/20 次...
  AvgF1 = 0.6461, NMI = 0.5279
运行第 16/20 次...
  AvgF1 = 0.7402, NMI = 0.5277
运行第 17/20 次...
  AvgF1 = 0.6419, NMI = 0.4738
运行第 18/20 次...
  AvgF1 = 0.6496, NMI = 0.5534
运行第 19/20 次...
  AvgF1 = 0.6684, NMI = 0.4192
运行第 20/20 次...
  AvgF1 = 0.6397, NMI = 0.4939
表格1已保存: PolBK_clustering_results_20runs.xlsx
表格2已保存: PolBK_metrics_20ru